# World Model (Dreamer-style) - FFAI Assistant

Encoder + Transition + Reward Prediction.

In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f"TensorFlow: {tf.__version__}")

In [ ]:
STATE_SIZE, ACTION_SIZE, LATENT_SIZE = 256, 15, 64

# Encoder: State -> Latent
encoder = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(STATE_SIZE,)),
    keras.layers.Dense(LATENT_SIZE, activation='relu')
], name='WM_Encoder')

# Transition: (Latent, Action) -> Next Latent
transition_input = keras.Input(shape=(LATENT_SIZE + ACTION_SIZE,))
x = keras.layers.Dense(128, activation='relu')(transition_input)
transition_output = keras.layers.Dense(LATENT_SIZE, activation='relu')(x)
transition = keras.Model(transition_input, transition_output, name='WM_Transition')

# Reward Predictor: (Latent, Action) -> Reward
reward_input = keras.Input(shape=(LATENT_SIZE + ACTION_SIZE,))
x = keras.layers.Dense(64, activation='relu')(reward_input)
reward_output = keras.layers.Dense(1)(x)
reward_pred = keras.Model(reward_input, reward_output, name='WM_Reward')

print(f"✓ Encoder: {encoder.count_params():,} params")
print(f"✓ Transition: {transition.count_params():,} params")
print(f"✓ Reward: {reward_pred.count_params():,} params")

In [ ]:
# Exportar
models = {
    'world_model_encoder': encoder,
    'world_model_transition': transition,
    'world_model_reward': reward_pred
}

for name, model in models.items():
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    with open(f'{name}.tflite', 'wb') as f:
        f.write(converter.convert())
    print(f"✓ Exportado: {name}.tflite")

In [ ]:
from google.colab import files
for name in models.keys():
    files.download(f'{name}.tflite')